# PROC QUANTREGによる寸法偏差上側裾のモデリング

## エグゼクティブサマリー

精密機械加工工場が気にかけるのは平均ではなく、**最悪ケース**の部品ごとの寸法偏差である。なぜなら、上側の裾がスクラップと顧客不良を左右するからだ。本ノートブックは **PROC QUANTREG** を用いて中央値と90パーセンタイルにおける分位点回帰を当てはめ、工具摩耗・サイクル速度・送り速度が中央値よりも高分位（スクラップリスク）の裾に対してはるかに強い影響を及ぼすことを明らかにする——これは工具が摩耗するにつれてばらつきが大きくなる不均一分散プロセスの特徴である。

## データソース

| データセット | 行数 | 説明 | 主な変数 |
|---------|------|-------------|---------------|
| `work.machining` | 100 | CNC旋盤ラインの合成された部品単位のレコードで、`call streaminit`/`rand`によりインラインで生成される。公称値からの寸法偏差（ミクロン単位）は、広がりが工具摩耗とサイクル速度とともに増大する不均一分散誤差でモデル化されているため、工程要因は中央値よりも上側の裾に対してより強く作用する。 | `Deviation`（応答変数、ミクロン）、`ToolWear`（累積切削時間）、`CycleSpeed`（rpm）、`CoolantTemp`（摂氏度）、`Humidity`（％RH）、`FeedRate`（mm/回転）、`Machine`（CLASS：M1〜M4）、`Shift`（CLASS：Day/Eve/Night）、`PartID` |

# 寸法偏差上側裾の工程要因モデリング

## 製造業ユースケース：CNC旋盤ラインにおけるスクラップリスクモデリング

精密機械加工ラインでは、公称値からの**寸法偏差**が大きくなりすぎると部品はスクラップとなる。工場が損失を被るのは*平均的な*部品ではなく、偏差分布の**上側の裾**である。通常の最小二乗回帰は条件付き平均をモデル化するため、事態が悪化したときにのみ重要になる要因を完全に見逃す可能性がある。

**分位点回帰**は、平均の代わりに選択した条件付き分位点（たとえば偏差の90パーセンタイル）をモデル化する。**PROC QUANTREG** は1回の呼び出しで1つまたは複数の分位点を当てはめ、各分位点における各要因のパラメータ推定値、標準誤差、信頼限界を報告する。以下を行う：

1. 誤差の広がりが工具摩耗とサイクル速度とともに増大する（そのため要因が中心よりも裾に強く作用する）現実的な合成部品単位データセットを生成する。
2. **中央値**（q = 0.50）と**スクラップリスクの裾**（q = 0.90）を1回の PROC QUANTREG 呼び出しで当てはめる。
3. 2つの係数セットを並べて比較し、中心から裾にかけて傾きがどのように変化するかを示す。
4. すべての部品に90パーセンタイル予測値でスコアを付け、アナリストが裾リスクの高い部品にフラグを立てられるようにする。

以下はすべて自己完結型である——外部ファイル、ネットワークともに不要。

## ステップ1 — 合成機械加工データの生成

4台の機械と3つのシフトにまたがる旋盤加工部品をシミュレートする。現実性を高める鍵となる工夫は**不均一分散**である：ランダム誤差項の標準偏差（`sigma`）は `ToolWear` と `CycleSpeed` とともに増大する。そのため、この2つの要因は `Deviation` の中央値よりも90パーセンタイルに対してはるかに強い影響を及ぼす——まさに分位点回帰がその真価を発揮する状況である。`Humidity` はデータ生成過程において実質的な信号を持たないため、観察対象となるプラセボ要因として機能する。

In [1]:
データ work.machining;
    呼出 streaminit(20260531);
    長さ Machine $2 Shift $5;
    繰返 PartID = 1 から 100;
        /* クラス変数 */
        m = rand('integer', 1, 4);
        Machine = cats('M', PUT(m, 1.));
        s = rand('integer', 1, 3);
        もし s = 1 なら Shift = 'Day';
        他 もし s = 2 なら Shift = 'Eve';
        他 Shift = 'Night';

        /* 連続工程要因 */
        ToolWear     = rand('uniform') * 120;            /* 累積切削時間（分） */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* 主軸回転数（rpm） */
        CoolantTemp  = 22 + rand('normal') * 3;          /* 摂氏温度 */
        Humidity     = 45 + rand('normal') * 8;          /* ％RH（プラセボ） */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/回転 */

        /* 号機オフセット：1台の機械はより高温で稼働する */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* 夜勤シフトはわずかにドリフトする */
        shiftoff = (Shift = 'Night') * 1.2;

        /* 偏差の位置（中心傾向、単位：ミクロン） */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* 不均一分散の広がり：摩耗と速度とともに裾が広がる */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        もし Deviation < 0 なら Deviation = 0;

        保持 PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        出力;
    終了;
実行;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


### 生データの分布を確認する

モデリングの前に、応答変数が右に裾を引いており、意味のある上側の裾を持つことを確認する——これがスクラップを左右する分布の部分である。PROC UNIVARIATE から直接、中央値と上側パーセンタイルを出力し、90パーセンタイルが中央値よりどれだけ高い位置にあるかを確認する。

In [2]:
処理 単変量 データ=work.machining NOPRINT;
    変数 Deviation;
    出力 out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
実行;

処理 印刷 データ=work.devpct noobs 見出;
    見出 Med = '偏差の中央値'
          P90 = '90パーセンタイル'
          P95 = '95パーセンタイル'
          P99 = '99パーセンタイル';
実行;


            偏差の中央値                90パーセンタイル                95パーセンタイル                99パーセンタイル
------------------  -----------------------  -----------------------  -----------------------
      8.7251490709            12.4132063767            13.5691645665            17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## ステップ2 — 中央値とスクラップリスクの裾を同時に当てはめる

PROC QUANTREG は1回の呼び出しで両方の分位点を当てはめる：`QUANTILE=0.5 0.90`。`CLASS` ステートメントはカテゴリ工程要因（`Machine`、`Shift`）を宣言し、`MODEL` ステートメントはすべての候補連続効果を列挙する。`CI=SPARSITY` を指定すると、スパース性関数推定量を用いて各係数の標準誤差と95％信頼帯を算出する。

2つの出力ブロックを前後比較として読み取る：最初のブロック（q = 0.50）は*典型的な*部品を表し、2番目のブロック（q = 0.90）は*スクラップになりやすい*部品を表す。`ToolWear`、`CycleSpeed`、`FeedRate` に注目してほしい——シミュレートされた誤差の広がりが摩耗と速度とともに増大するように作られているため、これらの傾きは上側分位点でより大きくなるはずである。

In [3]:
処理 quantreg データ=work.machining ci=sparsity;
    分類 Machine Shift;
    見出 Deviation = '寸法偏差（ミクロン）'
          Machine    = '号機'
          Shift      = 'シフト'
          ToolWear   = '工具摩耗（分）'
          CycleSpeed = '主軸回転数（rpm）'
          CoolantTemp = '冷却液温度（℃）'
          Humidity   = '湿度（％RH）'
          FeedRate   = '送り速度（mm/回転）';
    模型 Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
実行;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: 寸法偏差（ミクロン）

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
工具摩耗（分）               0.0240       0.0033       0.0174       0.0305
主軸回転数（rpm）           -0.0007       0.0008      -0.0022       0.0009
冷却液温度（℃）              0.4542       0.0395       0.3767       0.5316
湿度（％RH）               0.0575       0.0150       0.0281       0.0868
送り速度（mm/回転）          -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -11.4711
MACHINE M3            


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## ステップ3 — 中心と裾を並べて比較する

2つの係数ブロックを並行して読むのは扱いにくいため、`ODS OUTPUT ParameterEstimates=`（`Quantile` 列を含む）でパラメータ表を取得し、連続要因について q = 0.50 と q = 0.90 の推定値を1つの比較表にマージする。`Tail - Median` 列が見出しの数値であり、大きな正の値はその要因が典型的な部品を動かす以上にスクラップリスクの裾を強く押し上げることを意味する。

In [4]:
ODS 出力 ParameterEstimates=work.pe;
処理 quantreg データ=work.machining ci=sparsity;
    分類 Machine Shift;
    見出 Deviation = '寸法偏差（ミクロン）'
          ToolWear   = '工具摩耗（分）'
          CycleSpeed = '主軸回転数（rpm）'
          CoolantTemp = '冷却液温度（℃）'
          Humidity   = '湿度（％RH）'
          FeedRate   = '送り速度（mm/回転）';
    模型 Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
実行;

/* 連続要因ごとに中央値と裾の傾きをマージする */
データ work.compare;
    保持 Parameter MedianSlope TailSlope TailMinusMedian;
    結合
        work.pe(where=(Quantile=0.5)
                keep=Quantile Parameter Estimate
                rename=(Estimate=MedianSlope))
        work.pe(where=(Quantile=0.9)
                keep=Quantile Parameter Estimate
                rename=(Estimate=TailSlope));
    基準 Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
実行;

処理 印刷 データ=work.compare noobs 見出;
    見出 Parameter       = '要因'
          MedianSlope     = '傾き @ q=0.50'
          TailSlope       = '傾き @ q=0.90'
          TailMinusMedian = '裾 - 中央値';
    書式 MedianSlope TailSlope TailMinusMedian 10.4;
実行;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: 寸法偏差（ミクロン）

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
工具摩耗（分）               0.0146       0.0045       0.0057       0.0235
主軸回転数（rpm）           -0.0000       0.0011      -0.0021       0.0021
冷却液温度（℃）              0.4838       0.0528       0.3802       0.5873
湿度（％RH）               0.0678       0.0203       0.0280       0.1076
送り速度（mm/回転）          -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
工具摩耗（分）               0.0416       0.0021       0.0375       0.0457
主軸回転数（rpm）            0.0008       0.0005      -0.0002       0.0018
冷却液温度（℃）              0.3907       0.0245       0.3428       0.4387
湿度（％RH）               0.0807       0.0094       0.0623       0.0991
送り速度（mm/回転）           5.9122       3.6069      -1.1572      12.9817


         要因      傾き 


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## ステップ4 — 90パーセンタイルですべての部品にスコアを付ける

`OUTPUT` ステートメントは、90パーセンタイルの予測値を予測標準誤差（`STDP`）およびチェック損失残差とともに、部品ごとに1行ずつ書き出す。`ID` ステートメントで `PartID` を引き継ぎ、支配的な2つの要因をコピーして、アナリストがリスクの高い部品を上位にソートできるようにする。小さな PROC PRINT で最初にスコア付けされた部品を表示する。

In [5]:
処理 quantreg データ=work.machining ci=sparsity;
    分類 Machine Shift;
    id PartID;
    見出 Deviation = '寸法偏差（ミクロン）'
          Machine    = '号機'
          Shift      = 'シフト'
          ToolWear   = '工具摩耗（分）'
          CycleSpeed = '主軸回転数（rpm）'
          CoolantTemp = '冷却液温度（℃）'
          FeedRate   = '送り速度（mm/回転）';
    模型 Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    出力 out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
実行;

処理 印刷 データ=work.scored(obs=10) noobs 見出;
    変数 PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    見出 PartID  = '部品ID'
          Machine = '号機'
          ToolWear = '工具摩耗'
          CycleSpeed = '主軸回転数'
          PredP90 = '予測90パーセンタイル偏差'
          SEPred  = '予測標準誤差'
          Resid   = 'チェック損失残差';
実行;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: 寸法偏差（ミクロン）

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
工具摩耗（分）               0.0438       0.0012       0.0416       0.0461
主軸回転数（rpm）            0.0037       0.0003       0.0032       0.0043
冷却液温度（℃）              0.3377       0.0133       0.3116       0.3638
送り速度（mm/回転）          14.9464       2.0482      10.9321      18.9608


    部品ID            工具摩耗            主軸回転数                        予測90パーセンタイル偏差              予測標準誤差                  チェック損失残差
--------  --------------  -----


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## 結果の解釈

**裾は中心とは異なる物語を語る。** 2つの係数ブロック（ステップ2）とマージ表（ステップ3）を比較すると、`ToolWear`、`CycleSpeed`、`FeedRate` の傾きは中央値よりも90パーセンタイルで明らかに大きい。これはデータ生成メカニズムが可視化されたものである：誤差の広がりが摩耗と速度とともに増大するように構築したため、これらの要因は中央値の偏差をほとんど動かさないが、スクラップになりやすい上側の裾を強く膨張させる。平均に基づく回帰では、まさにスクラップにとって重要なレバーを過小評価していたはずである。

**`ToolWear` の信号が最も明瞭である。** その傾きは中央値フィット（0.015）から裾フィット（0.042）にかけておよそ3倍になり、90パーセンタイルの信頼帯はゼロを大きく上回る位置にある——摩耗は高裾リスクの最も信頼できる単一の要因である。`CycleSpeed` は中央値ではほぼ横ばいだが、裾では正に転じ、これは広がり項における役割と整合する。

**分位点回帰は位置と広がりを分離する。** `CoolantTemp` は位置項には入るが広がり項には入らないため、その傾きは両方の分位点で1度あたり0.4〜0.5ミクロン程度にとどまる——分布全体をシフトさせるのであって、裾を扇状に広げるわけではない。`Humidity` はデータ生成過程において実質的な信号を持たないが、わずか100部品では見かけ上の小さな傾きを拾ってしまう。その `Tail - Median` の変化（0.013）は `ToolWear` のそれ（0.027）より1桁小さく、`FeedRate` のそれ（12.3）に比べれば無視できるほど小さいため、裾の要因のようには振る舞わない。真に無関係な変数でも小標本ではゼロでないように見えることがあるというのが正直な教訓であり、まさにこれこそ、数千部品にわたるライセンス版のフル生産実行がこれらの推定値を引き締める理由である。

**運用上の効果。** `OUTPUT` の予測値は、すべての部品に標準誤差付きの90パーセンタイル偏差予測を与え、出荷前に裾リスクの高い部品にフラグを立てる。実行可能な結論は次のとおりである：厳しい公差のジョブを実行する際は工具交換間隔を短縮し、サイクル速度に上限を設けること。どちらの制御も、寸法偏差の上側裾（スクラップを引き起こす要因）に直接作用するからである。

*規模に関する注記：* このノートブックは無償版エンジン上で実行されており、各ステップは100観測に制限されているため、上記の傾きは100部品で推定されたものであり、裾のフィットはフル生産実行よりも必然的にノイズが多くなる。中心対裾のパターンはこの規模でもすでに確認できるが、部品ストリーム全体にわたるライセンス版の実行であれば、すべての信頼帯はさらに引き締まるはずである。